In [10]:
import torch
import gc
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass
from typing import List, Dict
import time
import json

In [13]:
sizes = [1024, 1024*1024, 100*1024*1024, 1024*1024*1024]  # 1KB, 1MB, 100MB, 1GB
size_labels = ["1 KB", "1 MB", "100 MB", "1 GB"]

print(f"{'Size':<15} {'cudaMalloc (ms)':<20} {'Tensor Creation (ms)':<25} {'Speedup':<10}")
print("-"*80)

for size, label in zip(sizes, size_labels):
    # Method 1: Direct CUDA malloc (simulated via torch.cuda.caching_allocator_alloc)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    _ = torch.empty(size // 4, dtype=torch.float32, device='cuda')
    torch.cuda.synchronize()
    cached_time = (time.perf_counter() - t0) * 1000
    
    # Clear cache to force real cudaMalloc
    torch.cuda.empty_cache()
    
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    tensor = torch.empty(size // 4, dtype=torch.float32, device='cuda')
    torch.cuda.synchronize()
    uncached_time = (time.perf_counter() - t0) * 1000
    
    del tensor

    # Method 2: Cached allocation (second allocation of same size)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    tensor2 = torch.empty(size // 4, dtype=torch.float32, device='cuda')
    torch.cuda.synchronize()
    reuse_time = (time.perf_counter() - t0) * 1000
    
    del tensor2
    
    speedup = uncached_time / reuse_time if reuse_time > 0 else 0
    print(f"{label:<15} {uncached_time:>18.3f}  {reuse_time:>23.3f}  {speedup:>8.1f}x")

torch.cuda.empty_cache()

Size            cudaMalloc (ms)      Tensor Creation (ms)      Speedup   
--------------------------------------------------------------------------------
1 KB                         0.010                    0.005       2.0x
1 MB                         0.983                    0.005     215.7x
100 MB                       2.098                    0.005     457.4x
1 GB                         2.077                    0.005     459.8x
